# Food-101 by Matyas Andrys

Training CNN on [Food-101 dataset on Kaggle](https://www.kaggle.com/datasets/ranitsarkar01/the-food-101-data-set) to classify food images.

You know... I'm hungry.

## Install and setup dependencies

In [ ]:
%pip install -q kagglehub matplotlib torch torchvision seaborn numpy pandas pillow

In [92]:
from pathlib import Path

import kagglehub as kaggle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
from torchvision import datasets, transforms

In [95]:
SEED = 42 # The meaning of life

np.random.seed(SEED)
torch.manual_seed(SEED)

sns.set_theme(style='whitegrid')

def get_best_device():
    if torch.cuda.is_available():
        return 'cuda'
    if torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'

device = torch.device(get_best_device())
print('torch version:', torch.__version__)
print('device:', device)

torch version: 2.14.1
device: mps


## Download the dataset

Use KaggleHub to download the dataset and then load the downloaded images.

In [87]:
food_dataset = datasets.ImageFolder(root=Path(kaggle.dataset_download('ranitsarkar01/the-food-101-data-set')) / 'food-101' / 'images')

## Exploration analysis

Let's look at the image dataset, mainly image sizes, color mode, and class distribution.

In [100]:
def print_dataset_info(dataset):
    rows = []

    # Load images directly from files to retrieve metadata only and avoid loading full images
    for image_path, label in dataset.samples:
        with Image.open(image_path) as image:
            rows.append({
                'width': image.width,
                'height': image.height,
                'label': dataset.classes[label],
                'mode': image.mode
            })

    data = pd.DataFrame(rows)
    display(data.describe())
    display(data.groupby('label').size().reset_index(name='image_count_per_class').describe())
    display(
        data.groupby('mode').size().to_frame('image_count')
        .sort_values('image_count', ascending=False)
    )

print_dataset_info(food_dataset)

,width,height
count,101000.000000,101000.000000
mean,495.791396,475.372129
std,45.674348,65.305769
min,193.000000,122.000000
25%,512.000000,452.000000
50%,512.000000,512.000000
75%,512.000000,512.000000
max,512.000000,512.000000


,image_count_per_class
count,101.0
mean,1000.0
std,0.0
min,1000.0
25%,1000.0
50%,1000.0
75%,1000.0
max,1000.0


,image_count
mode,
RGB,100997
L,3


As we can see, the dataset contains images of various sizes, mostly of 512 x 512.
Width goes from **193px** to **512px**. Height goes from **122px** to **512px**.
The most common mode is **RGB**, which is expected for color images. Three images are in **grayscale**.
The distribution of classes is consistent, with **1000** images per class and **101** classes. Total is **101000** images.